In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251016_175203.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/2500_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_2500_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_2500_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_2500_train_500_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id    subreddit                                              title  \
 0   n1i0pd      assault  I’m a male that was coerced, defrauded, assaul...   
 1  1lb4r4g  Miscarriage                        MMC in Australia. What now?   
 2  1lkcz7z  Miscarriage  8 week incomplete miscarriage, undecided next ...   
 3   aecuyf        metoo                                Help with an answer   
 4  1lgksxs    BabyBumps                             A Bit of Friend Drama…   
 
                                             selftext          created_utc  \
 0  Nearly 5 years ago, I was 20 years old, couldn...   2021-04-30 0:12:14   
 1  Hi everyone, I'm sorry to be here and I'm sorr...   2025-06-14 9:34:24   
 2  TW: describe symptoms of what I've experienced...  2025-06-25 18:08:24   
 3  I have a friend who says it’s not right that a...  2019-01-09 23:33:42   
 4  Buckle in, this will be a long one 😅\n\nMy bes...   2025-06-21 1:35:48   
 
                                                  url 

In [5]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r9_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r9_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r9_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r9_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

In [10]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [6]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r9_emb_A, r9_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_2500_train_500_test.json
